In [7]:
import os
import sys
import pickle
import gzip
import numpy as np
import tensorflow as tf
import multiprocessing
from matplotlib import rcParams

# ── Path setup ────────────────────────────────────────────────────────────
GLM_DIR = '/Users/akhilbandi/Documents/Portfolio/code_projects/Context Paper Code/GLM_encoding_model'
if GLM_DIR not in sys.path:
    sys.path.insert(0, GLM_DIR)

# ── Pipeline imports ───────────────────────────────────────────────────────
from core.session_pipeline import (
    extract_data,
    generate_predictors,
    stack_design_matrices,
    save_glm_data,
    open_glm_stacked_data,
    prepare_features_for_glm,
    fit_glm_single_session,
)

# ── Plot style ─────────────────────────────────────────────────────────────
rcParams['font.family'] = 'Arial'
rcParams['axes.spines.right'] = False
rcParams['axes.spines.top'] = False

print("✓ Imports complete")

✓ Imports complete


In [8]:
# ── Session registry ──────────────────────────────────────────────────────
# Keys: mouse_ID  |  Values: list of recording dates

PPC_AVC = {
    'IS-2-1L': ['2025-03-10', '2025-03-11', '2025-03-12', '2025-03-13',
                '2025-03-20', '2025-03-26', '2025-04-02', '2025-04-04', 
                '2025-04-08', '2025-04-09', '2025-05-05', '2025-05-27', 
                '2025-05-28', '2025-05-30', '2025-11-20'],
    'IS-3-1R': ['2024-12-23', '2024-12-24', '2024-12-27', '2024-12-28'],
    'JP-2-1L': ['2025-08-19', '2025-08-26', '2025-09-16', '2025-09-17',
                '2025-09-19', '2025-09-22'],
    'KA-1-00': ['2025-12-16', '2026-01-08'],
    'KA-2-00': ['2026-01-08', '2026-01-13', '2026-01-14', '2026-01-28'],
    'KO-5-1L': ['2026-03-05', '2026-03-06', '2026-03-12', '2026-03-13'],
    'KO-6-00': ['2026-03-12']
}

# IS-2-1L: '2025-04-14', '2025-04-15', '2025-04-22', '2025-05-06', '2025-05-08'

# PPC_AVC_test = {
#     'KO-5-1L': ['2026-03-05', '2026-03-06'],
# }

total = sum(len(v) for v in PPC_AVC.values())
print(f"Session registry: {len(PPC_AVC)} mice | {total} sessions total")

Session registry: 7 mice | 36 sessions total


In [9]:
def load_results(sessions_dict):
    """
    Load glm_results_combined.pkl for every session in sessions_dict.

    Returns
    -------
    sessions : list of (mouse_ID, date, results_dict)
    """
    sessions = []
    for mouse_ID, dates in sessions_dict.items():
        for date in dates:
            path = f'{BASE_DATA_PATH}/{mouse_ID}/{date}/glm_results_combined.pkl'
            if not os.path.exists(path):
                print(f'  [MISSING] {mouse_ID} / {date}')
                continue
            with open(path, 'rb') as f:
                res = pickle.load(f)
            sessions.append((mouse_ID, date, res))
            print(f'  [LOADED]  {mouse_ID} / {date}  ({res["cv_frac_dev"].shape[0]} neurons)')

    print(f'\nLoaded {len(sessions)} sessions.')
    return sessions

In [10]:
import gc
import pandas as pd

from core.glm_class import make_prediction, deviance
from core.session_pipeline import open_glm_stacked_data, prepare_features_for_glm, compute_trial_avg_r2

BASE_DATA_PATH = '/Volumes/Akhil Data/Akhil/ProcessedData'

def compute_group_contributions_session(res, X, Y, metadata):
    """
    Model breakdown for a single session.
    For each predictor group, zero out its weights and measure the drop
    in test frac dev and trial-averaged R² across all neurons.

    Parameters
    ----------
    res      : dict   — saved results from fit_glm_single_session
    X        : ndarray (n_frames, n_features) — z-scored design matrix
    Y        : ndarray (n_frames, n_neurons)  — neural data (×10 scaled)
    metadata : dict of ndarrays — frame-level metadata (context, choice,
               frame_in_trial, etc.), same length as X / Y

    Returns
    -------
    fd_full          : ndarray (n_neurons,)  — full model test frac dev
    r2_full          : ndarray (n_neurons,)  — full model test trial-avg R²
    contributions    : dict  group_name -> ndarray (n_neurons,)
                       drop in frac dev when group is ablated (positive = contributes)
    r2_contributions : dict  group_name -> ndarray (n_neurons,)
                       drop in trial-avg R² when group is ablated (positive = contributes)
    """
    w         = res['weights']               # (n_features, n_neurons)
    w0        = res['w0'].flatten()          # (n_neurons,)
    pred_info = res['predictor_info']        # split groups
    test_mask = res['combined_test_mask']

    X_test    = X[test_mask]
    Y_test    = Y[test_mask]
    meta_test = {k: v[test_mask] for k, v in metadata.items()}

    # Full model — frac dev
    Y_pred_full = make_prediction(X_test, w, w0, activation='exp')
    fd_full, _, _ = deviance(Y_pred_full, Y_test)
    fd_full = np.maximum(fd_full, 0.0)

    # Full model — trial-averaged R²
    r2_full = compute_trial_avg_r2(Y_test, Y_pred_full, meta_test)

    contributions    = {}
    r2_contributions = {}
    for group_name, info in pred_info.items():
        w_ablated = w.copy()
        w_ablated[info['start']:info['end'], :] = 0.0

        Y_pred_abl = make_prediction(X_test, w_ablated, w0, activation='exp')
        fd_abl, _, _ = deviance(Y_pred_abl, Y_test)
        fd_abl = np.maximum(fd_abl, 0.0)

        r2_abl = compute_trial_avg_r2(Y_test, Y_pred_abl, meta_test)

        contributions[group_name]    = fd_full - fd_abl    # drop in frac dev
        r2_contributions[group_name] = r2_full - r2_abl    # drop in trial-avg R²

    return fd_full, r2_full, contributions, r2_contributions


def compute_all_contributions(sessions):
    """
    Run model breakdown for all valid sessions.
    Loads X/Y from disk using the same pattern as run_all_sessions.

    Parameters
    ----------
    sessions : list of (mouse_ID, date, res)

    Returns
    -------
    df_contrib : DataFrame, indexed by neuron_uid
        Columns: mouse_ID, date, neuron_idx, test_frac_dev, test_trial_avg_r2,
                 contrib_<group> and contrib_r2_<group> for each predictor group
    """
    rows = []

    for mouse_ID, date, res in sessions:
        if res is None:
            print(f'  [SKIP] {mouse_ID} / {date} — no results')
            continue

        print(f'  Computing contributions: {mouse_ID} / {date} ...', end=' ')

        X = Y = None
        try:
            # Load X, Y, and metadata (applies ×10 Y scaling)
            X, Y, metadata, _, predictor_info, _ = open_glm_stacked_data(
                mouse_ID, date, print_summary=False
            )

            # Use the split predictor_info (audio_context/stim, visual_context/stim)
            # This matches what was used during fitting
            predictor_info_split, _ = prepare_features_for_glm(predictor_info)

            # Temporarily override pred_info in res with split version for breakdown
            res_with_split = {**res, 'predictor_info': predictor_info_split}

            fd_full, r2_full, contributions, r2_contributions = \
                compute_group_contributions_session(res_with_split, X, Y, metadata)

            n_neurons = len(fd_full)
            for n in range(n_neurons):
                row = {
                    'mouse_ID':          mouse_ID,
                    'date':              date,
                    'neuron_idx':        n,
                    'test_frac_dev':     fd_full[n],
                    'test_trial_avg_r2': r2_full[n],
                }
                for group_name, contrib in contributions.items():
                    row[f'contrib_{group_name}'] = contrib[n]
                for group_name, r2_contrib in r2_contributions.items():
                    row[f'contrib_r2_{group_name}'] = r2_contrib[n]
                rows.append(row)

            print(f'✓  ({n_neurons} neurons)')

        except Exception as e:
            import traceback
            print(f'✗ FAILED')
            traceback.print_exc()

        finally:
            del X, Y
            gc.collect()

    df_contrib = pd.DataFrame(rows)
    df_contrib['neuron_uid'] = (
        df_contrib['mouse_ID'] + '_' + df_contrib['date']
        + '_n' + df_contrib['neuron_idx'].astype(str)
    )
    df_contrib = df_contrib.set_index('neuron_uid')

    print(f'\nDone — {len(df_contrib):,} neurons across {len(sessions)} sessions')
    return df_contrib

In [11]:
sessions = load_results(PPC_AVC)

  [LOADED]  IS-2-1L / 2025-03-10  (2115 neurons)
  [LOADED]  IS-2-1L / 2025-03-11  (1958 neurons)
  [LOADED]  IS-2-1L / 2025-03-12  (1783 neurons)
  [LOADED]  IS-2-1L / 2025-03-13  (1143 neurons)
  [LOADED]  IS-2-1L / 2025-03-20  (1800 neurons)
  [LOADED]  IS-2-1L / 2025-03-26  (1757 neurons)
  [LOADED]  IS-2-1L / 2025-04-02  (1884 neurons)
  [LOADED]  IS-2-1L / 2025-04-04  (2019 neurons)
  [LOADED]  IS-2-1L / 2025-04-08  (2049 neurons)
  [LOADED]  IS-2-1L / 2025-04-09  (1598 neurons)
  [LOADED]  IS-2-1L / 2025-05-05  (2147 neurons)
  [LOADED]  IS-2-1L / 2025-05-27  (2571 neurons)
  [LOADED]  IS-2-1L / 2025-05-28  (1451 neurons)
  [LOADED]  IS-2-1L / 2025-05-30  (1122 neurons)
  [LOADED]  IS-2-1L / 2025-11-20  (1874 neurons)
  [LOADED]  IS-3-1R / 2024-12-23  (1855 neurons)
  [LOADED]  IS-3-1R / 2024-12-24  (2139 neurons)
  [LOADED]  IS-3-1R / 2024-12-27  (2742 neurons)
  [LOADED]  IS-3-1R / 2024-12-28  (2838 neurons)
  [LOADED]  JP-2-1L / 2025-08-19  (2325 neurons)
  [LOADED]  JP-2-1L 

In [12]:
df_contrib = compute_all_contributions(sessions)

  Computing contributions: IS-2-1L / 2025-03-10 ... Loading: /Volumes/Akhil Data/Akhil/ProcessedData/IS-2-1L/2025-03-10/glm_data_stacked.pkl
  ✓ Loaded  X: (68496, 150)  Y: (68496, 2115)
Predictor groups:
  position_only       : 10 features  (cols 0–9)
  audio_context       : 10 features  (cols 10–19)
  audio_stim          : 10 features  (cols 20–29)
  visual_context      : 10 features  (cols 30–39)
  visual_stim         : 10 features  (cols 40–49)
  velocity            : 20 features  (cols 50–69)
  choice_spatial      : 60 features  (cols 70–129)
  turn_onset          :  8 features  (cols 130–137)
  outcome             : 12 features  (cols 138–149)

Total: 150 features
✓  (2115 neurons)
  Computing contributions: IS-2-1L / 2025-03-11 ... Loading: /Volumes/Akhil Data/Akhil/ProcessedData/IS-2-1L/2025-03-11/glm_data_stacked.pkl
  ✓ Loaded  X: (59332, 150)  Y: (59332, 1958)
Predictor groups:
  position_only       : 10 features  (cols 0–9)
  audio_context       : 10 features  (cols 10–19)


/Users/akhilbandi/Documents/Portfolio/code_projects/Context Paper Code/GLM_encoding_model/session_pipeline.py:446: RuntimeWarning: divide by zero encountered in divide
  r2 = np.where(SS_tot > 1e-12, 1.0 - SS_res / SS_tot, 0.0)


✓  (2122 neurons)
  Computing contributions: KO-5-1L / 2026-03-05 ... Loading: /Volumes/Akhil Data/Akhil/ProcessedData/KO-5-1L/2026-03-05/glm_data_stacked.pkl
✗ FAILED
  Computing contributions: KO-5-1L / 2026-03-06 ... Loading: /Volumes/Akhil Data/Akhil/ProcessedData/KO-5-1L/2026-03-06/glm_data_stacked.pkl


Traceback (most recent call last):
  File "/var/folders/tb/ct0s7r6j0q90l_p7cs6hshr80000gn/T/ipykernel_73587/520189418.py", line 94, in compute_all_contributions
    X, Y, metadata, _, predictor_info, _ = open_glm_stacked_data(
                                           ^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/akhilbandi/Documents/Portfolio/code_projects/Context Paper Code/GLM_encoding_model/session_pipeline.py", line 108, in open_glm_stacked_data
    glm_data = pickle.load(f)
               ^^^^^^^^^^^^^^
EOFError: Ran out of input


  ✓ Loaded  X: (77782, 150)  Y: (77782, 863)
Predictor groups:
  position_only       : 10 features  (cols 0–9)
  audio_context       : 10 features  (cols 10–19)
  audio_stim          : 10 features  (cols 20–29)
  visual_context      : 10 features  (cols 30–39)
  visual_stim         : 10 features  (cols 40–49)
  velocity            : 20 features  (cols 50–69)
  choice_spatial      : 60 features  (cols 70–129)
  turn_onset          :  8 features  (cols 130–137)
  outcome             : 12 features  (cols 138–149)

Total: 150 features
✓  (863 neurons)
  Computing contributions: KO-5-1L / 2026-03-12 ... Loading: /Volumes/Akhil Data/Akhil/ProcessedData/KO-5-1L/2026-03-12/glm_data_stacked.pkl
  ✓ Loaded  X: (58958, 150)  Y: (58958, 1338)
Predictor groups:
  position_only       : 10 features  (cols 0–9)
  audio_context       : 10 features  (cols 10–19)
  audio_stim          : 10 features  (cols 20–29)
  visual_context      : 10 features  (cols 30–39)
  visual_stim         : 10 features  (cols 

In [14]:
save_path = os.path.join(GLM_DIR, 'df_contrib.pkl')
df_contrib.to_pickle(save_path)
print(f'Saved df_contrib → {save_path}  ({os.path.getsize(save_path)/1e6:.2f} MB)')

Saved df_contrib → /Users/akhilbandi/Documents/Portfolio/code_projects/Context Paper Code/GLM_encoding_model/df_contrib.pkl  (12.11 MB)


In [ ]:
# ── Usage ──────────────────────────────────────────────────────────────────────
# df_contrib = compute_all_contributions(sessions)
#
# Columns produced per neuron:
#   test_frac_dev          — full model Poisson frac dev on test set
#   test_trial_avg_r2      — full model trial-averaged R² on test set
#   contrib_<group>        — drop in frac dev when group is ablated
#   contrib_r2_<group>     — drop in trial-avg R² when group is ablated
#
# After running, merge with your existing neuron dataframe:
# drop_cols = ['mouse_ID', 'date', 'neuron_idx', 'test_frac_dev', 'test_trial_avg_r2']
# df_neurons = df_neurons.join(df_contrib.drop(columns=drop_cols), how='left')
#
# Dominant predictor by frac dev contribution:
# contrib_cols = [c for c in df_contrib.columns if c.startswith('contrib_') and not c.startswith('contrib_r2_')]
# df_contrib['dominant_predictor'] = df_contrib[contrib_cols].idxmax(axis=1).str.replace('contrib_', '')
#
# Dominant predictor by R² contribution:
# contrib_r2_cols = [c for c in df_contrib.columns if c.startswith('contrib_r2_')]
# df_contrib['dominant_predictor_r2'] = df_contrib[contrib_r2_cols].idxmax(axis=1).str.replace('contrib_r2_', '')